In [4]:
import numpy as np
import pandas as pd
import re
import string
import nltk

In [6]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation,'')
    return text

In [8]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [9]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [13]:
def preprocessing(text):
    data = pd.DataFrame([text], columns=['tweet'])
    
    data['tweet']  = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x, flags=re.MULTILINE) for x in x.split()))
    data["tweet"]= data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].str.replace('\d+', '', regex = True)
    data['tweet'] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"]= data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
C:\Users\A C E R\AppData\Local\Temp\ipykernel_8040\3694283837.py:7: SyntaxWarning: invalid escape sequence '\d'
  data["tweet"] = data["tweet"].str.replace('\d+', '', regex = True)


In [17]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [19]:
def vectorizer(ds, vocabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1

        vectorized_lst.append(sentence_lst)
    vectorized_lst_new = np.asarray(vectorized_lst, dtype=np.float32)

    return vectorized_lst_new

In [24]:
import pickle

In [25]:
with open('../static/model/model.pickle','rb') as f:
    model = pickle.load(f)

In [ ]:
def get_prediction(txt, tokens):
    preprocessed_txt = preprocessing(txt)
    vectorized_txt = vectorizer(preprocessed_txt, tokens)
    prediction = model.predict(vectorized_txt)
    if prediction == 1:
        return 'negative feedback'
    else:           
        return 'positive feedback'

In [ ]:
txt = "bad product. not recommend"
get_prediction(txt, tokens)

'negative feedback'

In [32]:
txt = "great product. I like it"
get_prediction(txt, tokens)

'positive feedback'